<a href="https://colab.research.google.com/github/frappedegansito/MachineLearning/blob/main/Unidad%203/P2U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
# Importamos librerias necesarias para regresion logistica y evaluacion de modelos
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [3]:
prestamos_df = pd.read_csv('prestamos_ok.csv')
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [6]:
X = prestamos_df [['funded_amnt', "int_rate", "grade", 'purpose', 'addr_state',
'home_ownership', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

KeyError: "['grade_code', 'purpose_code', 'addr_state_code', 'home_ownership_code'] not in index"

In [ ]:
# Dividimos el dataFrame df en df_train y df_test.
# 60% para dataset de entrenamient0, y 40% para dataset de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [ ]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Creamoos el clasificador de regresion logistica
clf1 = LogisticRegression(random_state=0)
# Entrenamos el clasificador
clf1.fit(X_train, y_train)

In [ ]:
# verificamos la precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento",
clf1.score(X_train, y_train) )


In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred = clf1.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
# Coeficiente de determinación
print( "Precisión:", clf1.score(X_test, y_test) )

In [ ]:
matriz_sin_balanceo = confusion_matrix(y_test, y_pred )
print(f'Matriz Confusion:\n', matriz_sin_balanceo)


In [ ]:
# Crear segundo clasificador con balanceo de clases
clf2 = LogisticRegression(random_state=0, class_weight="balanced")
# Entrenar el nuevo clasificador
clf2.fit(X_train, y_train)

In [ ]:
# verificamos la precisión del modelo 2 en la fase de entrenamiento
print("Exactitud del modelo de entrenamiento:", clf2.score(X_train, y_train))

In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred2 = clf2.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador: \n",
classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))
# Coeficiente de determinación
print( "Precisión:", clf2.score(X_test, y_test) )

In [ ]:
matriz_balanceo = confusion_matrix(y_test, y_pred2 )
print(f'Matriz Confusion:\n', matriz_balanceo)

In [ ]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clf2, X_test, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
"Variable": X.columns,
"Importancia Media": result.importances_mean,
"Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)


In [ ]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
error_y="Desviación", title="Importancia de características (Regresión Logística - Per
text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
y = prestamos_df["repaid"]
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
# Usamos los mismos parametros de tamaño y random usados anteriormente para que sea justa la compara
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.4, random_state=42, stratify=1)
# Normalizar variables (Regresión Logística (RL) es sensible a la escala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Entrenar modelo RL escalado
RL_scaled = LogisticRegression(random_state=0, class_weight='balanced')
RL_scaled.fit(X_train_scaled, y_train)
# Probar y evaluar el modelo RL escalado
y_pred_scaled = RL_scaled.predict(X_test_scaled)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))

In [ ]:
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_scaled))

In [ ]:
from sklearn.inspection import permutation_importance
result = permutation_importance(RL_scaled, X_test_scaled, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
"Variable": X.columns,
"Importancia Media": result.importances_mean,
"Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

In [ ]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
error_y="Desviación", title="Importancia de características (Regresión Logística - Permutation Importance",
text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()


In [ ]:
# Ajustar el umbral de decisión a 0.45
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.45
umbral = 0.45 # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.45:")
print(confusion_matrix(y_test, y_pred_umbral))


In [ ]:
# Ajustar el umbral de decisión a 0.40
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.40
umbral = 0.40 # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.40:")
print(confusion_matrix(y_test, y_pred_umbral))

In [ ]:
# Ajustar el umbral de decisión a 0.55
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.55
umbral = 0.55 # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.55")
print(confusion_matrix(y_test, y_pred_umbral))


In [ ]:
# Ajustar el umbral de decisión a 0.60
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.60
umbral = 0.60 # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.60")
print(confusion matrix(y test y pred umbral))

In [ ]:
# Debemos proporcionar 10 valores (variables predictoras)
yhat_predmanual = clf2.predict( [[5000, 15.5 , 5, 3, 4, 4, 30000, 12, 12, 0 ]] )